In [ ]:
import os
import time
from pathlib import Path
import jax
import jax.numpy as jp
import mujoco
from mujoco import mjx
import numpy as np
import viser
from mjviser import ViserMujocoScene

os.environ["XLA_PYTHON_CLIENT_PREALLOCATE"] = "false"

In [ ]:
path = Path.cwd() / "1d_pusher.xml"
model = mujoco.MjModel.from_xml_path(str(path))
data = mujoco.MjData(model)
mjx_model = mjx.put_model(model,
    contact_approximation="log barrier",
    contact_smoothing=1e2,
)
mjx_data = mjx.make_data(mjx_model)

server = viser.ViserServer()
server.scene.add_grid("grid")
scene = ViserMujocoScene(server, model, num_envs=1)
scene.camera_tracking_enabled = False

In [ ]:
mujoco.mj_kinematics(model, data)
scene.update_from_mjdata(data)

@jax.jit
def step(x, u):
    d = mjx_data.replace(qpos=x[:2], qvel=x[2:], ctrl=u)
    d = mjx.step(mjx_model, d)
    return jp.concatenate((d.qpos, d.qvel))

x = jp.zeros(model.nq + model.nv)
u = jp.array([0.0])

dt = model.opt.timestep
for t in jp.arange(0, 5, dt):
    u = u.at[0].set(0.2 * t)

    tic = time.time()
    x = step(x, u)
    print(f"\rStep took {(time.time() - tic) * 1e3:.3f} ms", end=" " * 10)

    data.qpos = np.array(x[:model.nq])
    mujoco.mj_kinematics(model, data)
    scene.update_from_mjdata(data)

    remain = model.opt.timestep - (time.time() - tic)
    if remain > 0:
        time.sleep(remain)